# Bronze — Central Bank of Iceland Ingestion
Fetches daily policy interest rates and monthly CPI from the Central Bank XML API.

**Source:** Seðlabanki Íslands XML Time Series API  
**Groups:** GroupID=1 (Policy Rates), GroupID=3 (CPI)  
**Output:** `bronze.central_bank_raw` (Delta table)

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import timedelta

BASE_URL = "https://sedlabanki.is/xmltimeseries/Default.aspx"
START_DATE_HISTORY = "1990-01-01"
REBUILD_HISTORY = True 

if spark.catalog.tableExists("bronze.central_bank_raw") and not REBUILD_HISTORY:
    last_date = spark.sql("SELECT MAX(CAST(TO_DATE(date_raw, 'M/d/yyyy h:mm:ss a') AS DATE)) FROM bronze.central_bank_raw").collect()[0][0]
    start_date = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")
    is_full_load = False
    print(f"Incremental load — fetching from {start_date}")
else:
    start_date = START_DATE_HISTORY
    is_full_load = True
    print(f"Full load — fetching history from {start_date}")

In [ ]:
def parse_central_bank_xml(xml_text):
    root = ET.fromstring(xml_text)
    records = []
    for ts in root.findall("TimeSeries"):
        ts_id, ts_name = ts.get("ID"), ts.find("Name").text
        for entry in ts.findall(".//Entry"):
            records.append({
                "group_name": root.find("Name").text,
                "series_id": ts_id,
                "series_name": ts_name,
                "date_raw": entry.find("Date").text,
                "value_raw": entry.find("Value").text
            })
    return pd.DataFrame(records)

In [ ]:
resp_rates = requests.get(BASE_URL, params={"DagsFra": start_date, "GroupID": 1, "Type": "xml"})
resp_cpi = requests.get(BASE_URL, params={"DagsFra": start_date, "GroupID": 3, "Type": "xml"})

df_rates = parse_central_bank_xml(resp_rates.text)
df_cpi = parse_central_bank_xml(resp_cpi.text)

new_data = pd.concat([df_rates, df_cpi], ignore_index=True)
new_data["ingested_at"] = pd.Timestamp.now()

spark_df = spark.createDataFrame(new_data)

if is_full_load:
    spark_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("bronze.central_bank_raw")
    print("Full history rebuild complete.")
else:
    spark_df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable("bronze.central_bank_raw")
    print(f"Appended {spark_df.count()} records.")